In [53]:
!pip install langchain langchain-core langchain-groq langchain-community

In [10]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv('ecommerce review.csv')
df.head()

,customer_id,customer_name,email,product_id,product_name,category,rating,review_title,review_text,review_date,helpful_votes,total_votes,feedback_category
0,CUST00001,Breanna Rodriguez,NaN,PROD1543,Camera,Electronics,3,Average product,Average product. Throughout method only find a...,2025-08-25,1000.0,5000,Neutral
1,CUST00002,Brian Jackson,andreachoi@example.net,PROD3825,Smartphone,Home,2,NaN,Packaging damaged. Size option yourself food p...,2025-08-29,14.0,37,Critical
2,CUST00003,Keith Smith,crystal54@example.org,PROD6963,Shoes,Accessories,4,Nice quality,Works well. Himself live left door among wide ...,2024-10-19,25.0,26,Positive
3,CUST00004,Betty Brooks,ehanson@example.net,PROD1677,Backpack,Home,5,Loved it,Excellent product. Pick three peace debate dis...,2025-01-24,16.0,50,Positive
4,CUST00005,Jacob Dennis,seansanders@example.org,PROD7783,Tablet,Electronics,5,Loved it,NaN,2025-03-20,21.0,25,Positive


In [12]:
df.isnull().sum()

,0
customer_id,0
customer_name,60
email,90
product_id,0
product_name,0
category,0
rating,0
review_title,60
review_text,60
review_date,0


In [13]:
# Detect count of  Duplicate rows
df.duplicated().sum()

np.int64(0)

In [14]:
df.dtypes

,0
customer_id,object
customer_name,object
email,object
product_id,object
product_name,object
category,object
rating,int64
review_title,object
review_text,object
review_date,object


In [15]:
cat_cols = df.dtypes[df.dtypes=='object'].index
print(cat_cols)

num_cols = df.dtypes[(df.dtypes=='int64') | (df.dtypes=='float64')].index
print(num_cols)

Index(['customer_id', 'customer_name', 'email', 'product_id', 'product_name',
       'category', 'review_title', 'review_text', 'review_date',
       'feedback_category'],
      dtype='object')
Index(['rating', 'helpful_votes', 'total_votes'], dtype='object')


In [16]:
for i in num_cols:
  df[i] = df[i].fillna(df[i].mean())

In [17]:
for i in cat_cols:
   if i == 'customer_name':
        df[i] = df[i].fillna('Other')
   else:
    df[i] = df[i].fillna(df[i].mode())

In [18]:
df.isnull().sum()

,0
customer_id,0
customer_name,0
email,89
product_id,0
product_name,0
category,0
rating,0
review_title,60
review_text,0
review_date,0


In [19]:
df = df.dropna(subset=['email', 'review_title'])

In [20]:
df.isnull().sum()

,0
customer_id,0
customer_name,0
email,0
product_id,0
product_name,0
category,0
rating,0
review_title,0
review_text,0
review_date,0


In [21]:
a1 = df[num_cols].describe(percentiles = [0.01,0.02,0.05,0.95,0.98,0.99]).T
a1 = a1.iloc[:, 3:]
a1

,min,1%,2%,5%,50%,95%,98%,99%,max
rating,1.0,1.0,1.0,1.0,3.0,5.0,5.0,5.0,5.0
helpful_votes,0.0,0.0,1.0,2.0,26.0,48.0,50.0,50.0,1000.0
total_votes,0.0,11.0,15.0,21.6,65.0,97.0,99.0,100.0,5000.0


In [22]:
df['helpful_votes'] = np.where(df['helpful_votes'] > 50.0, 50.0, df['helpful_votes'])



In [23]:
df['total_votes'] = np.where(df['total_votes'] > 100.0, 100.0, df['total_votes'])
df['total_votes'] = np.where(df['total_votes'] <11.0, 11.0, df['total_votes'])



In [24]:
a1 = df[num_cols].describe(percentiles = [0.01,0.02,0.05,0.95,0.98,0.99]).T
a1 = a1.iloc[:, 3:]
a1

,min,1%,2%,5%,50%,95%,98%,99%,max
rating,1.0,1.0,1.0,1.0,3.0,5.0,5.0,5.0,5.0
helpful_votes,0.0,0.0,1.0,2.0,26.0,48.0,50.0,50.0,50.0
total_votes,11.0,11.0,15.0,21.6,65.0,97.0,99.0,100.0,100.0


In [25]:
df['email'] = df['email'].str.replace('@', '', regex=False)

In [26]:
df['rating'].value_counts()

,count
rating,
1,599
2,591
5,574
4,568
3,521


In [27]:
df['Rating_Type'] = np.where(df['rating'].isin([1,2]),'Critical','Moderate or Good')


In [28]:
df['Rating_Type'].value_counts()

,count
Rating_Type,
Moderate or Good,1663
Critical,1190


In [29]:
critical_df = df[df['Rating_Type']=='Critical']
critical_df.shape

(1190, 14)

In [30]:
from collections import Counter

In [31]:
def most_common_complain_keywords(data):
  keyword_counter = Counter(data['review_title'])
  return keyword_counter

In [32]:
critical_keyword = most_common_complain_keywords(critical_df)
critical_keyword

Counter({'Not as expected': 120,
         'Packaging damaged': 116,
         'Arrived late': 138,
         'Bad experience': 99,
         'Broken on arrival': 112,
         'Very disappointed': 121,
         'Terrible quality': 111,
         'Waste of money': 131,
         'Poor quality': 118,
         'Not working': 124})

In [33]:
crit_keyword_df = pd.DataFrame(critical_keyword.items(),columns=['Keywords','Count'])
crit_keyword_df

,Keywords,Count
0,Not as expected,120
1,Packaging damaged,116
2,Arrived late,138
3,Bad experience,99
4,Broken on arrival,112
5,Very disappointed,121
6,Terrible quality,111
7,Waste of money,131
8,Poor quality,118
9,Not working,124


In [34]:
import plotly.express as px

In [35]:
fig = px.bar(crit_keyword_df,x='Keywords',y='Count',text='Count')
fig.show()

In [36]:
def topK_most_crit_reviews(data,K):
  top3_crit_reviews = data.sort_values(by='Count',ascending=False).head(K)
  return top3_crit_reviews

In [37]:
topK_most_crit_reviews(crit_keyword_df,3)

,Keywords,Count
2,Arrived late,138
7,Waste of money,131
9,Not working,124


In [49]:
def extract_cust_id_and_review(data,customer_id):
  text = data['review_title'][data['customer_id']==customer_id]
  return text


In [54]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import os

In [61]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [62]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.72,
    api_key=os.environ["GROQ_API_KEY"]
)

parser = StrOutputParser()

In [63]:
prompt  = PromptTemplate.from_template("""
Role
You are an experienced Customer Support Agent for an e-commerce company.

Task
Generate a short, personalized, professional, and empathetic apology email for the customer based on the provided Customer ID and Review.

Inputs
Customer ID: {Cust_ID}
Customer Review: {Review}
Instructions
Address the customer using their Customer ID.
Begin with a sincere apology.
Specifically acknowledge the complaint mentioned in the review. Do not generate a generic apology.
Show empathy and understanding.
Briefly assure the customer that the issue will be investigated and steps will be taken to prevent it from happening again.
If applicable, mention that the customer can contact the support team for additional assistance.
End the email with a polite closing and appreciation for the customer's feedback.
Keep the email between 80 and 120 words.
Maintain a warm, respectful, and professional tone.
Do not invent order details, dates, refund amounts, or promises that are not supported by the review.
Output Format

Subject: We're Sorry About Your Experience

Dear {Cust_ID},

<Generate a personalized apology email here based on the review.>

Best Regards,

Customer Support Team

""")

In [64]:
print(cat_cols)

Index(['customer_id', 'customer_name', 'email', 'product_id', 'product_name',
       'category', 'review_title', 'review_text', 'review_date',
       'feedback_category'],
      dtype='object')


In [65]:
chain = prompt | llm | parser

text_review = extract_cust_id_and_review(df,'CUST00001')
res = chain.invoke({'Cust_ID':'CUST00001','Review':text_review})
print(res)

Subject: We're Sorry About Your Experience

Dear CUST00001,

We apologize for the average product experience you had. We understand that our product did not meet your expectations, and for that, we are truly sorry. We acknowledge your disappointment and will investigate this issue to prevent it from happening again. If you need further assistance, please don't hesitate to contact us.

Best Regards,
Customer Support Team
